# ThermoNO on IC-ThermBench S2-S5 (report Sec 9.29)

**Question.** ThermoNO is exactly linear in the power map and nonlinear in geometry. On this
project's own data it beats a classical solver at the peak (Sec 9.26). Does the same inductive bias
hold on someone else's benchmark?

**Protocol.** IC-ThermBench's own split (10,800 train / 1,200 val / 3,000 test, unshuffled), their
vendored metric code, model selection on *their* val split only. S5 is scored zero-shot with the
S4 model, as in their paper. IC-ThermBench is a single-layer 64x64 grid with no layered stack, so
there is no classical backbone: ThermoNO predicts T - T_amb directly (`scripts/icb_thermono.py`).

**Numbers to compare against (test RMSE, K).** Therm-FM 0.443 / 0.716 / 0.933 (S2/S3/S4, best
published); SAU-FNO / U-FNO 0.703 / 0.802 / 1.216 (runner-up); our ridge-per-geom 1.943 / 2.488 /
3.203 (Sec 9.13). S5 zero-shot: Therm-FM 15.51, U-Net 19.10.

**Setup.** Attach the private dataset `rajulkabir/ic-thermbench-s2-s5`. Internet on. GPU T4 x1.

In [ ]:
import subprocess, sys, torch
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pyyaml'], check=True)

In [ ]:
import shutil, zipfile
from pathlib import Path
REPO = 'https://github.com/rajul-kk/thermo-3dic-surrogates.git'
CLONE, WORK = Path('/kaggle/working/repo'), Path('/kaggle/working')
OUT = WORK / 'icb_results'; OUT.mkdir(parents=True, exist_ok=True)
shutil.rmtree(CLONE, ignore_errors=True)
r = subprocess.run(['git', 'clone', '--depth', '1', REPO, str(CLONE)], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
print('repo at', subprocess.run(['git', '-C', str(CLONE), 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())

# Kaggle may or may not unpack the uploaded zip; handle both.
mats = list(Path('/kaggle/input').rglob('level2_steady/input.mat'))
if mats:
    DATA = mats[0].parent.parent
else:
    z = next(Path('/kaggle/input').rglob('s2_s5_datasets.zip'))
    zipfile.ZipFile(z).extractall(WORK / 'icb')
    DATA = next((WORK / 'icb').rglob('level2_steady/input.mat')).parent.parent
print('data root', DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
RUNS = [('level2', []), ('level3', []), ('level4', ['--transfer', 'level5'])]
EXTRA = ['--epochs', '100', '--patience', '15']      # defaults: ch 32, 4 blocks, 32 modes, lr 5e-3, plain MSE
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import time
for scope, more in RUNS:
    out = OUT / f'icb_thermono_{scope}_seed0.json'
    if out.exists():
        print(scope, 'already done'); continue
    t = time.time()
    cmd = [sys.executable, 'scripts/icb_thermono.py', '--scope', scope, '--data-root', str(DATA),
           '--device', DEVICE, '--out', str(out)] + more + EXTRA
    p = subprocess.Popen(cmd, cwd=CLONE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'Warning' not in line:
            print('  ' + line, end='', flush=True)
    assert p.wait() == 0, f'{scope} failed'
    print(f'  ({(time.time() - t) / 60:.1f} min)')

In [ ]:
import json
PUB = {'level2': (0.4427, 0.7028), 'level3': (0.7161, 0.8016), 'level4': (0.9334, 1.2158), 'level5': (15.51, 19.10)}
RIDGE = {'level2': 1.943, 'level3': 2.488, 'level4': 3.203}
rows = {}
for scope, _ in RUNS:
    f = OUT / f'icb_thermono_{scope}_seed0.json'
    if not f.exists(): continue
    r = json.loads(f.read_text())
    for sc in [k for k in r if k.startswith('level')]:
        rows[sc] = {**r[sc], 'n_params': r['n_params'], 'epochs': r['epochs_run'], 'minutes': r['minutes'],
                    'trained_on': scope}
print(f"{'scope':<8}{'ThermoNO':>10}{'Therm-FM':>10}{'runner-up':>11}{'ridge/geom':>11}{'peak err':>10}{'top50':>8}")
for sc, m in rows.items():
    print(f"{sc:<8}{m['rmse']:>10.4f}{PUB[sc][0]:>10.4f}{PUB[sc][1]:>11.4f}{RIDGE.get(sc, float('nan')):>11.3f}"
          f"{m['max_temperature_error']:>10.3f}{m['topk50_temperature_difference']:>8.3f}")
(OUT / 'summary.json').write_text(json.dumps(rows, indent=1))